In [ ]:
using JLD2
using LensFactory
using CairoMakie

In [ ]:
# Read samples
sample = jldopen("./SMACS0723/SMACS0723_sample.jld2")

# Read data
model  = sample["model"]
chains = sample["chains"]
logL   = sample["logL"]

# Lens redshift
z_d = model.observation.z_d

# Reference position
RA_REF  = model.observation.reference[1];
DEC_REF = model.observation.reference[2];

# Field of view (FOV)
FOV = model.observation.FOV;
pixel_scale = model.observation.pixel_scale

# Construct grid
x, y = Lenses.get_meshgrid(FOV[1]/2, FOV[2]/2, pixel_scale);

In [ ]:
# Get the best-fit cosmology
cosmo = LensModel.get_cosmology(sample);

# Source redshift
z_s = 1.5

Dol = Cosmology.angular_diameter_distance(cosmo, 0.0, z_d)
Dls = Cosmology.angular_diameter_distance(cosmo, z_d, z_s)
Dos = Cosmology.angular_diameter_distance(cosmo, 0.0, z_s)
adis = Dls/Dos

# Get the best-fit lens model
best_model, _ = LensModel.get_best_model(model; mcmc_chains=chains, mcmc_logL=logL, burn_in=0.0);

In [ ]:
# Calculate best-fit

In [ ]:
# Plot image plane for the best-fit lens model
fig, ax = Lenses.plot_image_plane(best_model, x, y, adis)
display(fig)

In [ ]:
# Plot magnification map for the best-fit lens model
fig, ax = Lenses.plot_magnification_map(best_model, x, y, adis; 
                                        plane=:image, 
                                        heatmap_kws=(colorrange=(1,50), colormap=:viridis, colorscale=log10))
display(fig)


In [ ]:
src = (1, 1)
time_delay = Lenses.get_time_delay(best_model, x, y, adis, z_d, Dol, src)
time_delay .= (time_delay .- minimum(time_delay)) ./ Constants.YEAR2SECOND

# Plot time delay
fig, ax = Lenses.plot_sky(FOV[1]/2, FOV[2]/3; figure_size=(500, 400))

cs = heatmap!(ax, x[:,1], y[1,:], time_delay, colormap=:viridis, colorrange=(minimum(time_delay), maximum(time_delay)))
cb = Colorbar(fig[1,2], cs; label=L"$t_d$", width=15, labelrotation=3π/2)

contour!(ax, x[:,1], y[1,:], time_delay, levels=append!(collect(0.01:1:50)), color=:white, linewidth=0.5)
display(fig)
     